# Generate Context Graphs

# Imports

In [1]:
import networkx as nx
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import duckdb
import random
import re
import itertools
from tqdm import tqdm as tqdm

# Directories

In [2]:
# Define current directory
cwd = Path.cwd()
# Define DATA directory
DATA = cwd.parents[1]/'data'/'canada'

# Define INPUT directory
INPUT = DATA / 'input'

# Define OUTPUT directory
OUTPUT = DATA / 'output'

# Define LEVEL5 directory
LEVEL5 = OUTPUT / 'level_5'

# Define CONTEXT directory
CONTEXT = OUTPUT / 'context_graphs'

# Define RNASEQ directory
RNASEQ = CONTEXT / 'rnaseq'

# Define CID directory
CID = CONTEXT / 'cid'

# Define MEAN directory
MEAN = CONTEXT / 'mean'

# Define NAME directory
NAME = CONTEXT / 'name'

# Functions

## General

In [3]:
def file_to_list(path):
    '''
    Converts a .txt file to a list
    '''

    with open(f'{path}', 'r', encoding = 'utf-8') as f:
        list_file = [line.strip() for line in f]
    
    return list_file

def list_to_file(path, data):
      '''
      Saves a list or set to a .txt file with no header.
      '''

      with open(path, 'w') as f:
            for item in sorted(data):
                  f.write(f'{item}\n')

def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()

## Extract LINCS data

In [4]:
def load_lincs_data(
        path_info,
        path_perturbagens,
        path_genes,
        path_landmark
) -> None:
    '''
    Loads all relevant lincs data for downstream analysis
    '''

    print('Loading relevant datasets...')
    # Load LINCS info
    df_info = pickle_load(path_info)
    print('LINCS info loaded')
    # Load perturbagen info
    df_perturbagen = pickle_load(path_perturbagens)
    print('Perturbagen info loaded')
    # Load gene info
    df_genes = pickle_load(path_genes)
    print('Gene info loaded')
    # Load landmark genes
    list_landmark = file_to_list(path_landmark)
    print('Landmark gene list loaded')

def filter_lincs_data(
        df_info,
        list_cells,
        list_timepoints,
        list_doses
) -> list:
    '''
    Takes LINCS info dataframe and filter variables to extract relevant LINCS signature IDs for data filtering.
    '''

    print('Filtering for LINCS signature IDs:')
    print(f'  > Cell line(s): {list_cells}')
    print(f'  > Timepoint(s): {list_timepoints}')
    print(f'  > Dose(s): {list_doses}')



def filter_lincs_data(
        path_data,
        path_info,
        path_genes,
        path_perturbagens,
        path_landmark,
        list_cells,
        list_timepoints,
        list_doses,
        num_drugs,
        list_specific
) -> pd.DataFrame:
    '''
    Takes various filter inputs to extract LINCS1000 expression data.
    '''

    print('Loading relevant datasets')
    print()

    # Load LINCS info
    df_info = pickle_load(path_info)
    # Load perturbagen info
    df_perturbagen = pickle_load(path_perturbagens)
    # Load gene info
    df_genes = pickle_load(path_genes)
    # Load landmark genes
    

# Context Graphs

## RNASeq Graphs

Level 5-equivalent experimental data calculated in `2_preprocess_rnaseq.ipynb` is extracted and overlayed onto the landmark graph.

In [5]:
# Define drug list
list_treatments = ['Halo', 'Nita', 'Paro']
# Load landmark graph
graph_landmark = pickle_load(OUTPUT / 'graph_landmark.pkl')

# Iterate through Level 5 data
for file in LEVEL5.iterdir():

    # Load data
    df = pd.read_csv(file)
    # Set index
    df.set_index('geneid', inplace = True)
    
    # Get filename
    filename = str(file).split('\\')[-1]
    timepoint = filename.split('_')[0]

    # Iterate over treatments
    for treatment in list_treatments:
        
        # Get treatment data
        df_treat = df[treatment]
        # Convert to dictionary
        dict_attributes = df_treat.to_dict()

        # Copy landmark graph
        graph = graph_landmark.copy()

        # Set node attributes
        nx.set_node_attributes(graph, dict_attributes, name = 'x')

        # Save graph
        pickle_save(RNASEQ / f'{timepoint.upper()}_{treatment}.pkl', graph)

In [6]:
print("RNASEQ directory:", RNASEQ.resolve())
print("Exists:", RNASEQ.exists())

RNASEQ directory: D:\DDesktop\_python\work\data\canada\output\context_graphs\rnaseq
Exists: True


## LINCS Graphs

Filtered signature data `df_signature_values.parquet`, previously generated in `1_preprocess_lincs.ipynb`, is queried without loading, avoiding large file import.

$N$ random perturbagens are extracted from this data, including any specific perturbagens. Dexp values are extracted and applied to copies of the landmark graph to generate context graphs.

In [7]:
# Load landmark graph
graph_landmark = pickle_load(OUTPUT / 'graph_landmark.pkl')

# Load statin list
list_statins = file_to_list(OUTPUT / 'list_statin_ids.txt')
# Load opioid list
list_opioids = file_to_list(OUTPUT / 'list_opioid_ids.txt')
# Load profen list
list_profens = file_to_list(OUTPUT / 'list_profen_ids.txt')
# Load hdac list
list_hdac = file_to_list(OUTPUT / 'list_hdac_ids.txt')

# Define parquest location
path_to_data = OUTPUT / 'df_signature_values.parquet'

In [8]:
# Define desired total number of perturbagens
NUM_PERTURBAGENS = 2000
# Define specific perturbagen list
list_specific_perturbagens = []

### Specific Compounds

Here, all data pertaining to pre-specified perturabagen types is extracted before randomly sampling data.

In [9]:
# Define list of lists
list_compounds = [list_statins, list_opioids, list_profens, list_hdac, list_specific_perturbagens]
list_labels = ['statin', 'opioid', '-profen', 'hdac inhibitor', 'specified']

# Initialise data
df_compounds = pd.DataFrame()

# Iterate over lists
for compound_list, label in zip(list_compounds, list_labels):

    if len(compound_list) == 0:
        print(f'No perturbagens for {label} compound types provided')

    else:
        # Generate tuple
        ids_tuple = tuple(compound_list)

        # Source data
        df = duckdb.connect().execute(f"""
            SELECT *
            FROM '{str(path_to_data)}'
            WHERE perturbagen_id IN {ids_tuple}
        """).df()

        # Concatenate data
        df_compounds = pd.concat([df_compounds, df])
        
        # Get unique IDs
        num_unique = len(pd.unique(df['perturbagen_id']))
        # Calculate percent
        percent_compounds = num_unique / len(compound_list) * 100
        print(f'{percent_compounds:.2f}% of {label} compounds found in signature data ({num_unique}/{len(compound_list)})')
print()

# Get updated list of all specified perturbagen IDs
list_specified = list(pd.unique(df_compounds['perturbagen_id']))
print(f'{len(list_specified)} total perturbagens from specified lists found in data')
# Show data
df_compounds.head()

61.90% of statin compounds found in signature data (13/21)
100.00% of opioid compounds found in signature data (12/12)
75.00% of -profen compounds found in signature data (9/12)
83.33% of hdac inhibitor compounds found in signature data (10/12)
No perturbagens for specified compound types provided

44 total perturbagens from specified lists found in data


,gene_id,lincs_name,lincs_desc,landmark,cid,dexp,perturbagen_id,dose,dataset,cell_line,timepoint,perturbagen_name
0,5720,PSME1,proteasome activator subunit 1,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,0.380776,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
1,466,ATF1,activating transcription factor 1,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,-0.427864,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
2,6009,RHEB,Ras homolog enriched in brain,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,-0.991012,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
3,2309,FOXO3,forkhead box O3,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,1.401294,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
4,387,RHOA,ras homolog family member A,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,0.194531,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin


### Random Compounds

All previously specified perturbagen IDs are excluded and the remainder of the total desired number of compounds are randomly sampled from the data.

In [10]:
# Calculate remaining perturbagen amount
NUM_RANDOM = NUM_PERTURBAGENS - len(list_specified)

print(f'Randomly sampling data for {NUM_RANDOM} remaining perturbagens')

# Get all perturbagen IDs from signature data
df_all = duckdb.connect().execute(f"""
    SELECT DISTINCT perturbagen_id
    FROM '{str(path_to_data)}'
""").df()

# Get remaining perturbagen IDs
df_remain = df_all[~df_all['perturbagen_id'].isin(list_specified)]
# Perturbagen IDs to sample
list_random = df_remain['perturbagen_id'].sample(n = NUM_RANDOM).tolist()
# Convert to tuple
random_tuple = tuple(list_random)

# Get data for sampled IDs
df_random = duckdb.connect().execute(f"""
            SELECT *
            FROM '{str(path_to_data)}'
            WHERE perturbagen_id IN {random_tuple}
""").df()

# Show data
df_random.head()

Randomly sampling data for 1956 remaining perturbagens


,gene_id,lincs_name,lincs_desc,landmark,cid,dexp,perturbagen_id,dose,dataset,cell_line,timepoint,perturbagen_name
0,5720,PSME1,proteasome activator subunit 1,1,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,-0.623000,BRD-A07824748,10,CPC005,HT29,6H,flavanone
1,466,ATF1,activating transcription factor 1,1,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,-1.483600,BRD-A07824748,10,CPC005,HT29,6H,flavanone
2,6009,RHEB,Ras homolog enriched in brain,1,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,-0.406733,BRD-A07824748,10,CPC005,HT29,6H,flavanone
3,2309,FOXO3,forkhead box O3,1,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,0.373000,BRD-A07824748,10,CPC005,HT29,6H,flavanone
4,387,RHOA,ras homolog family member A,1,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,-0.959167,BRD-A07824748,10,CPC005,HT29,6H,flavanone


### All Compounds

Combines extracted data for specified and randomly sampled compounds for context graphs.

In [11]:
# Concatenate data
df_context = pd.concat([df_compounds, df_random])
# Show data
df_context.head()

,gene_id,lincs_name,lincs_desc,landmark,cid,dexp,perturbagen_id,dose,dataset,cell_line,timepoint,perturbagen_name
0,5720,PSME1,proteasome activator subunit 1,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,0.380776,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
1,466,ATF1,activating transcription factor 1,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,-0.427864,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
2,6009,RHEB,Ras homolog enriched in brain,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,-0.991012,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
3,2309,FOXO3,forkhead box O3,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,1.401294,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin
4,387,RHOA,ras homolog family member A,1,CPC004_HT29_6H:BRD-K66296774-001-02-0:10,0.194531,BRD-K66296774,10,CPC004,HT29,6H,fluvastatin


### Gene Signal Per CID

Level 5 data contains consensus signal, but may contain multiple experiments/batches for the same set of filters. This is denoted by the individual CIDs.

It is possible to iterate over the unique CID IDs, and produce more than one graph per set of conditions. Graphs created will be tagged with all relevant metadata.

In [12]:
# Remove all existing files in CID directory
for file in CID.iterdir():
    file.unlink()

In [13]:
# # Get unique CIDs
# list_cids = pd.unique(df_context['cid'])

# # Iterate over CIDs
# for cid in tqdm(list_cids, desc='Generating CID-level context graphs', total = len(list_cids)):

#     # Slice data
#     df_slice = df_context[df_context['cid'] == cid].copy()

#     # Extract metadata
#     meta = df_slice.iloc[0]
#     perturbagen_id   = meta['perturbagen_id']
#     perturbagen_name = meta['perturbagen_name']
#     dose             = meta['dose']
#     timepoint        = meta['timepoint']
#     cell_line        = meta['cell_line']
#     dataset          = meta['dataset']

#     # Extract data
#     df_slice = df_slice[['lincs_name', 'dexp']].set_index('lincs_name')
#     # Rename column
#     df_slice.rename(columns = {'dexp' : 'x'}, inplace = True)

#     # Copy graph
#     graph = graph_landmark.copy()
#     # Assign node attributes
#     nx.set_node_attributes(graph, df_slice.to_dict(orient='index'))

#     # Format name
#     safe_cid = cid.replace(':', '_')

#     # Save graphs
#     pickle_save(CID / f'{safe_cid}_{perturbagen_id}_{cell_line}_{dose}_{timepoint}.pkl', graph)

### Generate Mean CID Signal

Rather than produce multiple context graphs per set of filters, average the signal of multiple CIDs for the same experimental parameters.

In [14]:
# Remove all existing files in MEAN directory
for file in MEAN.iterdir():
    file.unlink()

In [15]:
# # Get unique timepoints
# list_timepoints = pd.unique(df_context['timepoint'])
# # Get unique doses
# list_doses = pd.unique(df_context['dose']) 
# # Get unique perturbagen IDs
# list_perturbagens = pd.unique(df_context['perturbagen_id']) 
# # Get unique cell lines
# list_cells = pd.unique(df_context['cell_line']) 

# # Iterate over combinations of unique values
# for cell_line, perturbagen_id, dose, timepoint in tqdm(
#     itertools.product(list_cells, list_perturbagens, list_doses, list_timepoints),
#     desc = 'Generating mean signal perturbagen context graphs',
#     total = len(list_perturbagens)):

#     # Slice data
#     df_slice = df_context[
#         (df_context['cell_line'] == cell_line) &
#         (df_context['perturbagen_id'] == perturbagen_id) &
#         (df_context['dose'] == dose) &
#         (df_context['timepoint'] == timepoint)
#     ]

#     # Calculate mean
#     df_slice = df_slice[['lincs_name', 'dexp']].groupby(by = 'lincs_name').mean()
#     # Rename column
#     df_slice.rename(columns = {'dexp' : 'x'}, inplace = True)
#     # Copy landmark graph
#     graph = graph_landmark.copy()
#     # Set node attributes
#     nx.set_node_attributes(graph, df_slice.to_dict(orient = 'index'))
#     # Save graph
#     pickle_save(MEAN / f'{perturbagen_id}_{cell_line}_{dose}_{timepoint}.pkl', graph)

### Generate Name Signal

As multiple perturbagen IDs have been found to be linked to a single perturbagen name, context graphs are generated by averaging all signals of the same perturbagen name, for the same experimental parameters.

In [16]:
# Remove all existing files in NAME directory
for file in NAME.iterdir():
    file.unlink()

In [17]:
# Get unique timepoints
list_timepoints = pd.unique(df_context['timepoint'])
# Get unique doses
list_doses = pd.unique(df_context['dose']) 
# Get unique perturbagen IDs
list_perturbagens = pd.unique(df_context['perturbagen_name']) 
# Get unique cell lines
list_cells = pd.unique(df_context['cell_line'])

# Iterate over combinations of unique values
for cell_line, perturbagen_name, dose, timepoint in tqdm(
    itertools.product(list_cells, list_perturbagens, list_doses, list_timepoints),
    desc = 'Generating mean signal perturbagen context graphs',
    total = len(list_perturbagens)):

    # Slice data
    df_slice = df_context[
        (df_context['cell_line'] == cell_line) &
        (df_context['perturbagen_name'] == perturbagen_name) &
        (df_context['dose'] == dose) &
        (df_context['timepoint'] == timepoint)
    ]

    # Calculate mean
    df_slice = df_slice[['lincs_name', 'dexp']].groupby(by = 'lincs_name').mean()
    # Rename column
    df_slice.rename(columns = {'dexp' : 'x'}, inplace = True)
    # Copy landmark graph
    graph = graph_landmark.copy()
    # Set node attributes
    nx.set_node_attributes(graph, df_slice.to_dict(orient = 'index'))
    # Remove backslashes
    perturbagen_name = re.sub(r'[^A-Za-z0-9_\-]+', '', perturbagen_name)
    # Save graph
    pickle_save(NAME / f'{perturbagen_name}_{cell_line}_{dose}_{timepoint}.pkl', graph)

Generating mean signal perturbagen context graphs: 100%|██████████| 1940/1940 [1:50:32<00:00,  3.42s/it]
